# TinyGPT2 C4 Fit Reproduction Analysis

This notebook analyzes the pulled summary artifacts for
`tinygpt2_shuffle_dyck_c4_weight_transfer`.

Goals:
1. Compare the analytic fit blocks against the original pre-pretraining seed-derived subspace.
2. Plot downstream pretraining loss curves by initialization, using training tokens on the x-axis.


In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

%load_ext autoreload
%autoreload 2


def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / ".git").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src" / "platonic_init"))
sys.path.insert(0, str(REPO_ROOT / "src"))

import aesthetics as aes  # noqa: E402

sns = aes.sns

EXPERIMENT_NAME = "tinygpt2_shuffle_dyck_c4_weight_transfer"
EXPERIMENT_ROOT = REPO_ROOT / "artifacts" / "experiments" / EXPERIMENT_NAME
ANALYSIS_ROOT = EXPERIMENT_ROOT / "analysis"
BASIS_SWEEP_ROOT = ANALYSIS_ROOT / "basis_sweep"
PRETRAIN_ROOT = EXPERIMENT_ROOT / "pretraining"
FIGURE_ROOT = REPO_ROOT / "notebooks" / "figures" / EXPERIMENT_NAME

CONFIG_PATH = (
    REPO_ROOT / "configs" / "experiment_tinygpt2_shuffle_dyck_c4_weight_transfer.yaml"
)
config_payload = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
pre_cfg = config_payload["stages"]["prepretrain"]["training"]
TOKENS_PER_STEP = (
    int(pre_cfg["per_device_train_batch_size"])
    * int(pre_cfg.get("gradient_accumulation_steps", 1))
    * int(pre_cfg["block_size"])
)


def load_json(path: Path, default=None):
    if not path.exists():
        return {} if default is None else default
    return json.loads(path.read_text(encoding="utf-8"))


def parse_degree(label: str) -> int | None:
    m = re.search(r"_d(\d+)$", label)
    return int(m.group(1)) if m else None


def init_display_name(_family: str, name: str) -> str:
    if name == "random":
        return "Random"
    if name == "weight_transfer":
        return "Weight Transfer"
    degree = parse_degree(name)
    if degree is not None:
        return f"Plato (d={degree})"
    return name.replace("_", " ").title()


def init_sort_key(name: str) -> tuple[int, int | str]:
    if name == "random":
        return (0, 0)
    if name == "weight_transfer":
        return (1, 0)
    degree = parse_degree(name)
    if degree is not None:
        return (2, degree)
    return (3, name)

## Load Experiment Summaries

This notebook reads only lightweight JSON summaries, so it can run locally without remote checkpoints.


In [ ]:
weight_summary = load_json(ANALYSIS_ROOT / "weight_subspace_summary.json")
fit_manifest = load_json(BASIS_SWEEP_ROOT / "fit_blocks.json")
init_eval = load_json(PRETRAIN_ROOT / "init_eval.json", default={"results": []})
init_eval_curves = load_json(
    PRETRAIN_ROOT / "init_eval_basis_curves.json", default={"results": []}
)

fit_rows = []
for report_path in sorted(BASIS_SWEEP_ROOT.glob("analytic_fit_report_*.json")):
    slug = report_path.stem.replace("analytic_fit_report_", "")
    report = load_json(report_path)
    meta = fit_manifest.get(slug, {})
    fit_rows.append(
        {
            "fit_name": slug,
            "basis_type": meta.get("basis_type", "unknown"),
            "degree": parse_degree(slug),
            "mean_relative_error": report.get("mean_relative_error"),
            "num_tensors": len(report.get("tensors", {})),
            "report": report,
        }
    )
fit_df = (
    pd.DataFrame(fit_rows)
    .sort_values(["degree", "fit_name"], na_position="last")
    .reset_index(drop=True)
)
fit_name_order = fit_df["fit_name"].tolist()
fit_display_order = [init_display_name("plato", name) for name in fit_name_order]
fit_df["fit_name"] = pd.Categorical(
    fit_df["fit_name"], categories=fit_name_order, ordered=True
)
fit_df["display_name"] = pd.Categorical(
    fit_display_order, categories=fit_display_order, ordered=True
)
fit_df["degree_label"] = fit_df["degree"].astype("Int64").astype(str)

aes.set_palette(
    key="initializations",
    family="baseline",
    color="Greys",
    names=["random"],
    display_name=init_display_name,
)
aes.update_palette(
    key="initializations",
    family="transfer",
    color="Greens",
    names=["weight_transfer"],
    display_name=init_display_name,
)
aes.update_palette(
    key="initializations",
    family="plato",
    color="Oranges",
    names=list(reversed(fit_name_order)),
    display_name=init_display_name,
)
INIT_PALETTE = dict(aes.PALETTES["initializations"])
INIT_LABEL_ORDER = ["Random", "Weight Transfer", *fit_display_order]
tensor_rows = []
for tensor_name, info in weight_summary.get("tensors", {}).items():
    evr = info.get("explained_variance_ratio", [])
    tensor_rows.append(
        {
            "tensor": tensor_name,
            "numel": info.get("numel"),
            "evr_1": evr[0] if len(evr) > 0 else None,
            "evr_2": evr[1] if len(evr) > 1 else None,
            "evr_cum_2": sum(evr[:2]) if evr else None,
        }
    )
seed_df = (
    pd.DataFrame(tensor_rows)
    .sort_values("evr_cum_2", ascending=False)
    .reset_index(drop=True)
)
seed_evr_map = (
    seed_df.set_index("tensor")["evr_cum_2"]
    if not seed_df.empty
    else pd.Series(dtype=float)
)

summary_results = init_eval.get("results", [])
curve_results = init_eval_curves.get("results") or summary_results

print(f"Experiment root: {EXPERIMENT_ROOT}")
print(f"Tokens per training step: {TOKENS_PER_STEP:,}")
summary_count = len(summary_results)
print(f"Found {len(fit_df)} fit reports and {summary_count} downstream run summaries.")
display(
    fit_df[
        ["display_name", "basis_type", "degree", "mean_relative_error", "num_tensors"]
    ]
)
display(seed_df.head(10))

## Fit Blocks vs Original Pre-Pretraining Seeds

The analytic fit reports are computed on the PCA components extracted from the original pre-pretraining seeds.
Lower reconstruction error means a fit block better matches the shared seed-derived subspace.


In [ ]:
seed_overview = pd.DataFrame(
    {
        "stat": [
            "num_tensors",
            "total_params_analyzed",
            "median_evr_cum_2",
            "mean_evr_cum_2",
        ],
        "value": [
            weight_summary.get("num_tensors"),
            weight_summary.get("total_params_analyzed"),
            seed_df["evr_cum_2"].median() if not seed_df.empty else None,
            seed_df["evr_cum_2"].mean() if not seed_df.empty else None,
        ],
    }
)
display(seed_overview)
display(fit_df[["display_name", "degree", "mean_relative_error", "num_tensors"]])

In [ ]:
fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN * 1.9))

with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
    ax = fig.add_subplot(1, 1, 1)
    sns.barplot(
        data=fit_df,
        x="degree_label",
        y="mean_relative_error",
        hue="display_name",
        hue_order=fit_display_order,
        palette=INIT_PALETTE,
        legend=False,
        ax=ax,
    )
    ax.set_title("Analytic Fit Error vs Seed-Derived Subspace")
    ax.set_xlabel("Chebyshev Initialization Degree")
    ax.set_ylabel("Mean component relative error")
    ax.tick_params(axis="x", rotation=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

aes.save_figure(FIGURE_ROOT / "tinygpt2_fit_error_vs_degree", fig=fig)
plt.show()

**Note**
The token/position embedding tensors (`wte`/`wpe`) are excluded from the heatmap below. They are persistently hard to fit, but that is less informative here because embeddings are reset/reprojected before downstream pretraining.


In [ ]:
tensor_error_rows = []
for _, row in fit_df.iterrows():
    for tensor_name, info in row["report"].get("tensors", {}).items():
        tensor_error_rows.append(
            {
                "fit_name": row["fit_name"],
                "degree": row["degree"],
                "tensor": tensor_name,
                "mean_component_relative_error": info.get(
                    "mean_component_relative_error"
                ),
                "seed_evr_cum_2": seed_evr_map.get(tensor_name),
            }
        )

tensor_error_df = pd.DataFrame(tensor_error_rows)
non_embedding_tensor_error_df = tensor_error_df[
    ~tensor_error_df["tensor"].str.contains(r"wte|wpe", regex=True)
].copy()
hardest_tensors = (
    non_embedding_tensor_error_df.groupby("tensor")["mean_component_relative_error"]
    .max()
    .sort_values(ascending=False)
    .head(15)
    .index
)
degree_lookup = fit_df.set_index("fit_name")["degree"].to_dict()
heatmap_df = (
    non_embedding_tensor_error_df[
        non_embedding_tensor_error_df["tensor"].isin(hardest_tensors)
    ]
    .pivot(index="tensor", columns="fit_name", values="mean_component_relative_error")
    .reindex(columns=fit_name_order)
    .loc[list(hardest_tensors)]
)
heatmap_df.columns = [str(int(degree_lookup[name])) for name in heatmap_df.columns]

fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLE_ROW_IN * 1.5))
with sns.plotting_context("paper", font_scale=0.9, rc=aes.rcs):
    ax = fig.add_subplot(1, 1, 1)
    sns.heatmap(heatmap_df, cmap="magma_r", vmin=0.0, vmax=1.0, ax=ax)
    ax.set_title("Hardest Seed-Space Tensors by Fit Block")
    ax.set_xlabel("Chebyshev Initialization Degree")
    ax.set_ylabel("Tensor")
    ax.tick_params(axis="x", rotation=0)

aes.save_figure(FIGURE_ROOT / "tinygpt2_hardest_seed_space_tensors", fig=fig)
plt.show()

## Downstream Pretraining Loss Curves by Initialization

This section plots the available downstream eval-loss curves from `pretraining/init_eval*.json`,
with one series per initialization label and training tokens on the x-axis.


In [ ]:
result_rows = []
for row in summary_results:
    raw_label = row.get("label", row.get("variant", "unknown"))
    result_rows.append(
        {
            "label": init_display_name("", raw_label),
            "raw_label": raw_label,
            "variant": row.get("variant"),
            "init_mode": row.get("init_mode"),
            "best_eval_loss": row.get("best_eval_loss"),
            "final_eval_loss": row.get("final_eval_loss"),
            "train_loss": row.get("train_loss"),
        }
    )
result_df = (
    pd.DataFrame(result_rows).sort_values("best_eval_loss").reset_index(drop=True)
)
display(result_df)

curve_rows = []
for row in curve_results:
    raw_label = row.get("label", row.get("variant", "unknown"))
    display_label = init_display_name("", raw_label)
    for point in row.get("eval_curve", []):
        step = point.get("step")
        curve_rows.append(
            {
                "label": display_label,
                "raw_label": raw_label,
                "step": step,
                "tokens": step * TOKENS_PER_STEP if step is not None else None,
                "loss": point.get("eval_loss"),
                "metric": "eval_loss",
                "basis": row.get("basis"),
                "init_mode": row.get("init_mode"),
            }
        )
    for point in row.get("train_curve") or []:
        step = point.get("step")
        curve_rows.append(
            {
                "label": display_label,
                "raw_label": raw_label,
                "step": step,
                "tokens": step * TOKENS_PER_STEP if step is not None else None,
                "loss": point.get("loss"),
                "metric": "train_loss",
                "basis": row.get("basis"),
                "init_mode": row.get("init_mode"),
            }
        )

curves_df = pd.DataFrame(curve_rows)
available_labels = (
    sorted(curves_df.label.unique().tolist()) if not curves_df.empty else []
)
print(f"Available run labels: {available_labels}")
available_metrics = (
    sorted(curves_df.metric.unique().tolist()) if not curves_df.empty else []
)
print(f"Available metrics: {available_metrics}")
display(curves_df.head())

In [ ]:
if curves_df.empty:
    raise RuntimeError(
        "No downstream pretraining curves were found in the pulled summaries."
    )

metric_to_plot = "eval_loss"
selected_labels = ["Random", "Weight Transfer", "Plato (d=6)", "Plato (d=24)"]
plot_df = curves_df[curves_df["metric"] == metric_to_plot].copy()
plot_df = plot_df[plot_df["label"].isin(selected_labels)].copy()
label_order = [label for label in selected_labels if label in plot_df["label"].unique()]
fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLE_ROW_IN))
with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
    ax = fig.add_subplot(1, 1, 1)
    sns.lineplot(
        data=plot_df,
        x="tokens",
        y="loss",
        hue="label",
        hue_order=label_order,
        palette=INIT_PALETTE,
        linewidth=2.0,
        ax=ax,
    )
    ax.set_title("Loss by Initialization (TinyGPT-2)")
    ax.set_xlabel("Training tokens")
    ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)
    ax.set_ylabel("Validation loss (C4)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(title=None, loc="upper right", frameon=False)

aes.save_figure(
    FIGURE_ROOT / "tinygpt2_downstream_eval_loss_selected_initializations",
    fig=fig,
    save_png=False,
)
plt.show()

## Notes

- The fit-quality section is tied directly to the original pre-pretraining seeds because the analytic fit reports are computed on the seed-derived PCA subspace.
- The downstream section plots whatever run summaries are currently present on disk. If only `random` and `weight_transfer` are available so far, the notebook will reflect that.
- Once fit-based downstream runs finish and their summaries are synced, rerunning this notebook should extend the curve plot automatically.
